# Analytical Mantle Flow: Educational Workflow

This notebook is a guided walkthrough of the modeling logic in this repository. It begins with two simple idealized geometries so you can build intuition, then scales up to a global plate-boundary case from Holt and Royden (2020).


## 0) Setup and Run Mode

- `RUN_HEAVY = False`: use the precomputed outputs already shipped in the repo (fast, no long reruns).
- `RUN_HEAVY = True`: rerun the scripts from inputs to regenerate outputs (slower, useful for full reproducibility).

Tip: start with `False`, then switch to `True` once you understand the workflow.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'flow_computations').exists() else CWD.parent
if not (ROOT / 'flow_computations').exists():
    for parent in CWD.parents:
        if (parent / 'flow_computations').exists():
            ROOT = parent
            break
if not (ROOT / 'flow_computations').exists():
    raise RuntimeError('Could not locate repository root containing flow_computations/.')

FLOW = ROOT / 'flow_computations'
RUN_HEAVY = False

print('Repo root:', ROOT)
print('RUN_HEAVY =', RUN_HEAVY)


In [ ]:
def run(cmd, cwd=FLOW, env=None):
    run_env = os.environ.copy() if env is None else dict(env)
    run_env.setdefault('MPLCONFIGDIR', '/tmp/mplcfg')
    run_env.setdefault('XDG_CACHE_HOME', '/tmp')
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(cwd), check=True, env=run_env)

def show_image(path, title=None):
    path = Path(path)
    if not path.exists():
        print('Missing image:', path)
        return
    img = plt.imread(path)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(title or path.name)
    plt.show()

def summarize_dp(path):
    arr = np.loadtxt(path)
    vec = arr[:,4]
    print(f"rows={vec.shape[0]}, mean={np.mean(vec):.3f}, median={np.median(vec):.3f}, min={np.min(vec):.3f}, max={np.max(vec):.3f}")


## 1) Simple Case A: Idealized Retreating Trench

This case is a clean baseline geometry. Use it to see what the pressure solver produces without additional geometric complexity.

What to look for:
- overall sign and magnitude of across-slab pressure discontinuity (`DP`)
- spatial pressure pattern in the figure

In [ ]:
if RUN_HEAVY:
    run([sys.executable, 'global_pressure_withPressurePlot.py',
         'LargeSP_RetreatingTrench', '3.0e20', '0', '500000', '0', '0', 'Subgrd.inp', '4.0e20'])

dp_a_path = FLOW / 'text_files' / 'LargeSP_RetreatingTrench.3e+20noslabflux' / 'DP.txt'
summarize_dp(dp_a_path)
show_image(FLOW / 'plots' / 'pressure_fields' / 'LargeSP_RetreatingTrench.3e+20noslabflux.plotvisc4e+20.png',
          title='Simple Case A: LargeSP_RetreatingTrench')


## 2) Simple Case B: Slab-Gap Geometry

This introduces a slab-gap-style geometry so you can compare how a structural change alters pressure/discontinuity behavior.

Suggested comparison with Case A:
- change in `DP` summary stats
- qualitative differences in pressure field structure

In [ ]:
if RUN_HEAVY:
    run([sys.executable, 'global_pressure_withPressurePlot.py',
         'LargeSP_RetreatingTrenchSlabGap', '3.0e20', '0', '500000', '0', '0', 'Subgrd.inp', '4.0e20'])

dp_b_path = FLOW / 'text_files' / 'LargeSP_RetreatingTrenchSlabGap.3e+20noslabflux' / 'DP.txt'
summarize_dp(dp_b_path)
show_image(FLOW / 'plots' / 'pressure_fields' / 'LargeSP_RetreatingTrenchSlabGap.3e+20noslabflux.plotvisc4e+20.png',
          title='Simple Case B: LargeSP_RetreatingTrenchSlabGap')


## 3) Global Case: From Pressure to Dip Comparison

Now we run the full global workflow (or load precomputed outputs):
1. compute global pressure and across-slab `DP`
2. compute subducting plate ages
3. compare modeled dips vs observed dip catalogue

This is the key end-to-end scientific path connecting model physics to observation-based evaluation.

In [ ]:
global_model = 'Slab2.0Final_NoJapTail_nnr_FS'
global_text = FLOW / 'text_files' / f'{global_model}.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux'
global_plot = FLOW / 'plots' / 'pressure_fields' / f'{global_model}.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux.plotvisc4e+20.png'
global_dip_plot = FLOW / 'plots' / 'dip_comparisons' / f'{global_model}.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux.fact1.327.png'

env = os.environ.copy()
env['DIPS_OBS_TXT'] = str(ROOT / 'dip_observations' / 'dip_catalogues' / 'Slab2_const-depth' / 'AllDips.txt')

if RUN_HEAVY:
    run([sys.executable, 'global_pressure_withPressurePlot.py', global_model, '3.0e20', '2', '500000', '0', '1', 'Subgrd.inp', '4.0e20'], env=env)
    run([sys.executable, 'get_SPages.py', global_model], env=env)
    run([sys.executable, 'plot_DipComparison_varyDPfactor.py', global_model, '3.0e20', '2', '500000', '0', '1', 'Subgrd.inp', '2', '12', '4', '5'], env=env)

summarize_dp(global_text / 'DP.txt')
show_image(global_plot, title='Global Pressure Field')
show_image(global_dip_plot, title='Global Modeled vs Observed Dip Comparison')


## 4) Interpretation Prompts

Use these questions to guide interpretation:
- How do `DP` magnitude/range shift from simple geometries to global geometry?
- Which regions appear to drive the largest modeled-vs-observed dip mismatches?
- If you vary viscosity/flux settings, which diagnostics change most strongly?

This helps connect code output to physical intuition rather than treating the notebook as only a button-click workflow.